# CO3133 A2 - Food-101 training on Kaggle (M3: every reported run)

1. *Settings → Accelerator:* a GPU (RTX 6000, T4, P100 all work), *Internet: On*.
2. *Add input →* the pre-resized dataset `food-101-256` (output of `kaggle_prepare.ipynb`, or an uploaded copy of
   `data/resized/food-101-256`). It must contain `meta/classes.txt` and `images/<class>/<id>.jpg`.
3. Optional, to **resume**: add the output of an earlier session of this notebook as input; its `outputs/runs` is
   copied back, finished runs are skipped and unfinished ones continue from `last.pt`.
4. Choose `SESSION` below (`"all"`, or `"A"` mandatory / `"B"` optional to split over two sessions or two accounts),
   then *Save Version → Save & Run All (Commit)*. Every run is given only the time left before `SESSION_HOURS`,
   so the session ends cleanly before Kaggle's 12 h limit; rerun with step 3 to finish what is left.
5. Download from the output tab: `results_bundle_*.zip` (run folders without weights) and
   `checkpoints_food101.zip` (best.pt of every finished run). Unzip both **at the repo root**.


In [ ]:
import os, subprocess, sys, time
T0 = time.time()
REPO_URL = "https://github.com/pbt245/CO3113-A2.git"
REF = "main"            # commit hash or tag of the final runs (recorded in every run's env.json)
REPO = "/kaggle/working/CO3113-A2"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "checkout", REF], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi
!nproc; free -g | head -2
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
import glob
def find_food101(pattern_root="/kaggle/input"):
    """Return every folder under /kaggle/input that has meta/classes.txt and images/."""
    hits = [os.path.dirname(os.path.dirname(p)) for p in glob.glob(f"{pattern_root}/**/meta/classes.txt", recursive=True)]
    return [h for h in hits if os.path.isdir(os.path.join(h, "images"))]
roots = [r for r in find_food101() if "256" in r] or find_food101()
IMAGE_ROOT = roots[0]
print("images:", IMAGE_ROOT)

# resume: copy outputs/runs of an earlier session (if added as input)
import shutil
for prev in glob.glob("/kaggle/input/**/outputs/runs/food101", recursive=True):
    print("restoring", prev)
    shutil.copytree(prev, "outputs/runs/food101", dirs_exist_ok=True)

In [ ]:
# what to run. Order = priority (mandatory first). E3 and P3 use seed 42 only (proposal section 5).
SESSION = "all"            # "all" | "A" (mandatory) | "B" (optional)
SESSION_HOURS = 11.0       # stop starting / continuing runs after this (Kaggle kills the session at 12 h)
SEEDS3 = [42, 123, 2026]
JOBS_A = [(e, s) for e in ["b0_simple_cnn", "p1_resnet50", "p2_efficientnet_b0", "e1_resnet50_linear_probe"] for s in SEEDS3]
JOBS_B = ([("e2_resnet50_label_smoothing", s) for s in SEEDS3]
          + [(e, 42) for e in ["e3_p1_resnet50_frac10", "e3_p1_resnet50_frac25", "e3_p1_resnet50_frac50",
                               "e3_b0_simple_cnn_frac10", "e3_b0_simple_cnn_frac25", "e3_b0_simple_cnn_frac50",
                               "p3_convnext_tiny"]])
JOBS = {"all": JOBS_A + JOBS_B, "A": JOBS_A, "B": JOBS_B}[SESSION]
NUM_WORKERS = os.cpu_count()      # data loading (JPEG decode + augmentation) is the bottleneck on fast GPUs
COMMON = [f"data.image_root={IMAGE_ROOT}", f"data.num_workers={NUM_WORKERS}", "train.show_progress=false"]
print(len(JOBS), "jobs, workers:", NUM_WORKERS)

In [ ]:
# one run_experiments call per job: finished runs are skipped, unfinished ones resumed; each run only gets the
# time left in the session (train.max_hours stops it cleanly after an epoch, keeping last.pt for the next session)
status = []
for exp, seed in JOBS:
    left = SESSION_HOURS - (time.time() - T0) / 3600
    if left < 0.4:
        status.append((exp, seed, "not started (session budget used)"))
        continue
    rc = subprocess.run([sys.executable, "scripts/run_experiments.py", "--models", exp, "--seeds", str(seed),
                         "--set", *COMMON, f"train.max_hours={left - 0.25:.2f}"]).returncode
    done = os.path.exists(f"outputs/runs/food101/{exp}/seed{seed}/eval/test_metrics.json")
    status.append((exp, seed, "finished" if done else f"UNFINISHED (rc {rc}) - rerun to resume"))
for s in status:
    print(*s)

In [ ]:
# free space: last.pt (weights + optimizer state) is only needed by unfinished runs
for f in glob.glob("outputs/runs/food101/*/seed*/checkpoints/last.pt"):
    if os.path.exists(os.path.join(os.path.dirname(os.path.dirname(f)), "eval", "test_metrics.json")):
        os.remove(f)
!python scripts/compare.py
!python scripts/pack_results.py --out-dir /kaggle/working
!python scripts/pack_results.py --checkpoints-only --out-dir /kaggle/working
!ls -lh /kaggle/working